In [2]:
!pip install -q wfdb
import wfdb, numpy as np, pandas as pd
wfdb.set_db_index_url('https://physionet-open.s3.amazonaws.com/')

base = 'https://physionet.org/files/challenge-2017/1.0.0/training/'
pn_kok = 'challenge-2017/1.0.0/training/'

sinyaller = {}   # kayıt adı -> 1B sinyal (x)
meta = []        # kayıt adı, etiket, küme, fs, süre

for dosya, etiket in [('RECORDS-af', 1), ('RECORDS-normal', 0)]:
    liste = pd.read_csv('https://physionet-open.s3.amazonaws.com/challenge-2017/1.0.0/training/' + dosya, header=None)[0].iloc[:200]
    for i, satir in enumerate(liste):
        klasor, kayit = satir.split('/')
        rec = wfdb.rdrecord(kayit, pn_dir=pn_kok + klasor)
        x = rec.p_signal[:, 0]
        sinyaller[kayit] = x
        meta.append({'kayit': kayit, 'etiket': etiket,
                     'kume': 'gelistirme' if i < 100 else 'test',
                     'fs': rec.fs, 'sure_s': len(x) / rec.fs})

meta = pd.DataFrame(meta)
print('Kayıt sayısı:', len(sinyaller), '(400 olmalı)')
print('fs değerleri:', meta['fs'].unique(), '(300 olmalı)')
print('En kısa süre (s):', meta['sure_s'].min())
print('En uzun süre (s):', meta['sure_s'].max())
print(pd.crosstab(meta['kume'], meta['etiket']))

Kayıt sayısı: 400 (400 olmalı)
fs değerleri: [300] (300 olmalı)
En kısa süre (s): 9.986666666666666
En uzun süre (s): 60.56666666666667
etiket        0    1
kume                
gelistirme  100  100
test        100  100


In [4]:
import numpy as np, pandas as pd
from scipy.signal import butter, sosfiltfilt, find_peaks

FS = 300
SOS = butter(2, [5, 15], btype='bandpass', fs=FS, output='sos')  # 5–15 Hz bant geçiren, 2. derece
MIN_ARA = int(0.25 * FS)      # iki R tepesi arası en az 0.25 s
HEIGHT_CARPANI = 0.4          # eşik = 0.4 × |süzülmüş sinyalin| 99. yüzdeliği
MIN_RR = 5                    # bundan az R–R aralığı varsa karar verilemedi

def r_tepeleri(x):
    f = np.abs(sosfiltfilt(SOS, x))   # sıfır faz için filtfilt, sonra mutlak değer
    h = HEIGHT_CARPANI * np.percentile(f, 99)
    tepe, _ = find_peaks(f, height=h, distance=MIN_ARA)
    return tepe

def olc(x):
    t = r_tepeleri(x)
    rr = np.diff(t) / FS
    if len(rr) < MIN_RR:
        return np.nan, np.nan, len(t)
    cv = rr.std(ddof=1) / rr.mean()
    med = np.median(rr)
    mdr = np.median(np.abs(rr - med)) / med
    return cv, mdr, len(t)

sonuc = [olc(sinyaller[k]) for k in meta['kayit']]
meta['cv'], meta['mdr'], meta['n_tepe'] = zip(*sonuc)

def hata_say(p, y):
    p = p.copy(); b = p == -1
    p[b] = 1 - y[b]                 # karar verilemedi = yanlış tahmin
    return p

def f1(y, p):                       # pozitif sınıf = AF (1)
    tp = ((y == 1) & (p == 1)).sum(); fp = ((y == 0) & (p == 1)).sum()
    fn = ((y == 1) & (p == 0)).sum()
    return 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) > 0 else 0.0

def tahmin(skor, esik):             # skor > eşik -> AF; NaN -> -1
    return np.where(np.isnan(skor), -1, (skor > esik).astype(int))

def esik_sec(skor, y):              # yalnız geliştirme verisiyle F1'i en büyük yapan eşik
    en_iyi = (-1.0, None)
    for e in np.unique(skor[~np.isnan(skor)]):
        f = f1(y, hata_say(tahmin(skor, e), y))
        if f > en_iyi[0]:
            en_iyi = (f, e)
    return en_iyi[1], en_iyi[0]

gel = meta[meta['kume'] == 'gelistirme']; tst = meta[meta['kume'] == 'test']
print('Küme kesişimi (kayıt adı):', len(set(gel['kayit']) & set(tst['kayit'])), '(0 olmalı)')
print('R tepesi sayısı (min/medyan/maks):', meta['n_tepe'].min(), meta['n_tepe'].median(), meta['n_tepe'].max())

y_gel = gel['etiket'].values; y_test = tst['etiket'].values
esik_cv, f_gel_cv = esik_sec(gel['cv'].values, y_gel)
esik_mdr, f_gel_mdr = esik_sec(gel['mdr'].values, y_gel)
tahmin_cv = tahmin(tst['cv'].values, esik_cv)
tahmin_mdr = tahmin(tst['mdr'].values, esik_mdr)

for ad, e, fg, p in [('CV', esik_cv, f_gel_cv, tahmin_cv), ('MDR', esik_mdr, f_gel_mdr, tahmin_mdr)]:
    print(f'{ad}: eşik (geliştirmede seçildi) = {e:.4f}, geliştirme F1 = {fg:.3f}, '
          f'test F1 = {f1(y_test, hata_say(p, y_test)):.3f}, '
          f'test karar verilemedi = {(p == -1).sum()}')

Küme kesişimi (kayıt adı): 0 (0 olmalı)
R tepesi sayısı (min/medyan/maks): 5 40.0 152
CV: eşik (geliştirmede seçildi) = 0.0836, geliştirme F1 = 0.760, test F1 = 0.747, test karar verilemedi = 1
MDR: eşik (geliştirmede seçildi) = 0.0448, geliştirme F1 = 0.844, test F1 = 0.826, test karar verilemedi = 1
